# Analyse HSIC-ANOVA hiérarchique sur sorties MAELIA réelles

Ce notebook applique la méthode HSIC-ANOVA hiérarchique ajoutée par Paul Saves aux sorties réellement produites par MAELIA.

Il ne dépend pas de `smt_generation.py` et n'utilise pas la sortie synthétique `yt` de ce fichier. Les entrées viennent du fichier `dataset_metamodel.csv` exporté par les simulations `terrainSA`, avec les colonnes `feat_0` à `feat_25` et les sorties `N_lixi`, `dCorg`, `rdt`.

La logique de `simulations/maelia_analysis.py` est conservée :

- normalisation des variables continues ;
- apprentissage d'une Random Forest par sortie ;
- importance par permutation pour construire les échelles de noyaux `theta` ;
- appel à `hsic_anova_hierarchical` avec les variables actives/inactives du plan hiérarchique.

Note pratique : HSIC construit des matrices de distance en `n × n`. Pour éviter une explosion mémoire sur 10 000 simulations, le notebook contient un paramètre `HSIC_MAX_SAMPLES`. Mets-le à `None` seulement si tu veux forcer le calcul complet et que la machine a assez de mémoire.

## 1. Imports, chemins et paramètres

In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path('/Users/benjamin/files/Repositories/Sensitivity_analysis_MAELIA')
ANALYSIS_DIR = PROJECT_ROOT / 'analysis'
TOOLS_DIR = ANALYSIS_DIR / 'tools'
OUTPUT_DIR = ANALYSIS_DIR / 'hsic_anova_results'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if str(TOOLS_DIR) not in sys.path:
    sys.path.insert(0, str(TOOLS_DIR))

from hsic_methods import hsic_anova_hierarchical

OUTPUT_COLS = ['N_lixi', 'dCorg', 'rdt']
DATASET_FILENAME = 'dataset_metamodel.csv'
DATASET_CANDIDATES = [
    PROJECT_ROOT / 'simulations' / 'log_terrainSA',
    PROJECT_ROOT / 'data' / DATASET_FILENAME,
    ANALYSIS_DIR / DATASET_FILENAME,
]

RANDOM_SEED = 42
RF_N_ESTIMATORS = 200
RF_MAX_DEPTH = 10
RF_MIN_SAMPLES_LEAF = 15
PERMUTATION_REPEATS = 10
THETA_SCALE = 5.0
THETA_IMPORTANCE_THRESHOLD = 0.005
HSIC_MAX_ORDER = 4

# Mettre None pour utiliser tous les points. Attention : coût mémoire en O(n² × p).
HSIC_MAX_SAMPLES = 1500

print(f'Projet : {PROJECT_ROOT}')
print(f'Outils HSIC : {TOOLS_DIR}')
print(f'Sorties : {OUTPUT_DIR}')


## 2. Chargement du dataset réel MAELIA

In [ ]:
def resolve_dataset_path(candidates, filename=DATASET_FILENAME):
    checked = []
    for candidate in candidates:
        candidate = Path(candidate).expanduser()
        checked.append(candidate)

        if candidate.is_file():
            return candidate, checked

        if candidate.is_dir():
            direct_csv = candidate / filename
            checked.append(direct_csv)
            if direct_csv.is_file():
                return direct_csv, checked

            nested = sorted(candidate.rglob(filename))
            checked.extend(nested[:5])
            if nested:
                return nested[0], checked

    return None, checked


DATASET_PATH, checked_dataset_paths = resolve_dataset_path(DATASET_CANDIDATES)

if DATASET_PATH is None:
    print('Aucun dataset terrainSA trouvé. Chemins testés :')
    for candidate in checked_dataset_paths:
        print(' -', candidate)
    raise FileNotFoundError(
        'Exécuter le notebook de simulation terrainSA jusqu’à export de dataset_metamodel.csv, '
        'ou renseigner DATASET_CANDIDATES avec le dossier contenant ce fichier.'
    )

df_raw = pd.read_csv(DATASET_PATH)
print('Dataset chargé :', DATASET_PATH)
print('Dimensions :', df_raw.shape)
display(df_raw.head())

feat_cols = [f'feat_{i}' for i in range(26)]
missing_features = [col for col in feat_cols if col not in df_raw.columns]
missing_outputs = [col for col in OUTPUT_COLS if col not in df_raw.columns]

if missing_features:
    raise ValueError(
        'Le dataset doit contenir les colonnes feat_0...feat_25. '
        f'Colonnes manquantes : {missing_features}'
    )
if missing_outputs:
    raise ValueError(f'Sorties MAELIA absentes du dataset : {missing_outputs}')


## 3. Reconstruction des 26 paramètres agricoles

In [ ]:
AGRI_FEATURES = [
    'n_ferti', 'has_prepa', 'nb_prepa', 'prepa_1',
    'nb_f1', 'type_f1_1', 'nb_f2', 'type_f2_1', 'nb_f3', 'type_f3_1',
    'prepa_2', 'type_f1_2', 'type_f2_2', 'type_f3_2',
    'Date_Semis', 'Delta_PREPA_Semis', 'Date_F1', 'Date_F2', 'Date_F3', 'Date_Recolte',
    'Dose_F1_1', 'Dose_F1_2', 'Dose_F2_1', 'Dose_F2_2', 'Dose_F3_1', 'Dose_F3_2',
]

AGRI_CATEGORICAL = [
    'n_ferti', 'has_prepa', 'nb_prepa', 'prepa_1',
    'nb_f1', 'type_f1_1', 'nb_f2', 'type_f2_1', 'nb_f3', 'type_f3_1',
    'prepa_2', 'type_f1_2', 'type_f2_2', 'type_f3_2',
]
AGRI_CONTINUOUS = [feature for feature in AGRI_FEATURES if feature not in AGRI_CATEGORICAL]

X_params = df_raw[feat_cols].copy()
X_params.columns = AGRI_FEATURES
for col in AGRI_FEATURES:
    X_params[col] = pd.to_numeric(X_params[col], errors='coerce')

Y_outputs = df_raw[OUTPUT_COLS].copy()
for col in OUTPUT_COLS:
    Y_outputs[col] = pd.to_numeric(Y_outputs[col], errors='coerce')

df = pd.concat([X_params, Y_outputs], axis=1).dropna(subset=OUTPUT_COLS).reset_index(drop=True)
print('Nombre de lignes exploitables :', len(df))
print('Paramètres agricoles :', len(AGRI_FEATURES))
print('Sorties :', OUTPUT_COLS)
display(df[AGRI_FEATURES + OUTPUT_COLS].head())


## 4. Variables actives/inactives du plan hiérarchique

La méthode HSIC doit savoir si une variable est active pour chaque simulation. Ici, cette information est reconstruite depuis la logique du plan SMT réellement utilisé : par exemple `Date_F2` n'est active que si `n_ferti >= 2`, et `Dose_F1_2` n'est active que si le premier apport contient deux produits.

In [ ]:
FEATURE_INDEX = {name: i for i, name in enumerate(AGRI_FEATURES)}

CONDITIONALLY_ACTING = [
    'nb_prepa', 'prepa_1', 'prepa_2', 'Delta_PREPA_Semis',
    'nb_f1', 'type_f1_1', 'type_f1_2', 'Date_F1', 'Dose_F1_1', 'Dose_F1_2',
    'nb_f2', 'type_f2_1', 'type_f2_2', 'Date_F2', 'Dose_F2_1', 'Dose_F2_2',
    'nb_f3', 'type_f3_1', 'type_f3_2', 'Date_F3', 'Dose_F3_1', 'Dose_F3_2',
]

num_is_decreed = np.array([feature in CONDITIONALLY_ACTING for feature in AGRI_FEATURES], dtype=bool)
is_categorical = np.array([feature in AGRI_CATEGORICAL for feature in AGRI_FEATURES], dtype=bool)


def build_acting_matrix(X_df):
    acting = np.ones((len(X_df), len(AGRI_FEATURES)), dtype=bool)

    n_ferti = X_df['n_ferti'].to_numpy()
    has_prepa = X_df['has_prepa'].to_numpy() == 1
    nb_prepa = X_df['nb_prepa'].to_numpy()
    nb_f1 = X_df['nb_f1'].to_numpy()
    nb_f2 = X_df['nb_f2'].to_numpy()
    nb_f3 = X_df['nb_f3'].to_numpy()

    def set_active(feature, mask):
        acting[:, FEATURE_INDEX[feature]] = mask

    for feature in ['nb_prepa', 'prepa_1', 'Delta_PREPA_Semis']:
        set_active(feature, has_prepa)
    set_active('prepa_2', has_prepa & (nb_prepa == 1))

    for feature in ['nb_f1', 'type_f1_1', 'Date_F1', 'Dose_F1_1']:
        set_active(feature, n_ferti >= 1)
    for feature in ['type_f1_2', 'Dose_F1_2']:
        set_active(feature, (n_ferti >= 1) & (nb_f1 == 1))

    for feature in ['nb_f2', 'type_f2_1', 'Date_F2', 'Dose_F2_1']:
        set_active(feature, n_ferti >= 2)
    for feature in ['type_f2_2', 'Dose_F2_2']:
        set_active(feature, (n_ferti >= 2) & (nb_f2 == 1))

    for feature in ['nb_f3', 'type_f3_1', 'Date_F3', 'Dose_F3_1']:
        set_active(feature, n_ferti >= 3)
    for feature in ['type_f3_2', 'Dose_F3_2']:
        set_active(feature, (n_ferti >= 3) & (nb_f3 == 1))

    return acting


x_is_acting_full = build_acting_matrix(df[AGRI_FEATURES])
acting_summary = pd.DataFrame({
    'variable': AGRI_FEATURES,
    'conditionnelle': num_is_decreed,
    'categorielle': is_categorical,
    'frequence_active': x_is_acting_full.mean(axis=0),
})
display(acting_summary)
acting_summary.to_csv(OUTPUT_DIR / 'hsic_variable_acting_summary.csv', index=False)


## 5. Normalisation numérique

Les variables continues sont ramenées dans `[0, 1]` sur les bornes observées dans le dataset réel. Les variables catégorielles restent sous leur codage entier, car `hsic_methods.py` leur applique une distance d'égalité/différence.

In [ ]:
X_numeric_full = df[AGRI_FEATURES].copy()

for col in AGRI_FEATURES:
    X_numeric_full[col] = pd.to_numeric(X_numeric_full[col], errors='coerce')

for col in AGRI_CATEGORICAL:
    X_numeric_full[col] = X_numeric_full[col].fillna(0).round().astype(float)

# Pour les catégorielles conditionnelles, on force une modalité inactive dédiée.
# Sinon une variable inactive codée 0 pourrait être confondue avec une vraie modalité active 0.
for col in AGRI_CATEGORICAL:
    idx = FEATURE_INDEX[col]
    if num_is_decreed[idx]:
        X_numeric_full.loc[~x_is_acting_full[:, idx], col] = -1.0

for col in AGRI_CONTINUOUS:
    values = X_numeric_full[col].astype(float)
    fill_value = values.median()
    values = values.fillna(fill_value)
    vmin = values.min()
    vmax = values.max()
    if np.isfinite(vmin) and np.isfinite(vmax) and vmax > vmin:
        X_numeric_full[col] = (values - vmin) / (vmax - vmin)
    else:
        X_numeric_full[col] = 0.0

X_numeric_full = X_numeric_full.to_numpy(dtype=float)
print('Matrice X normalisée :', X_numeric_full.shape)


## 6. Sous-échantillon HSIC reproductible

La Random Forest est entraînée sur toutes les lignes disponibles. Le calcul HSIC peut ensuite être réalisé sur un sous-échantillon reproductible pour rester compatible avec la mémoire disponible.

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
n_total = len(df)

if HSIC_MAX_SAMPLES is None or HSIC_MAX_SAMPLES >= n_total:
    hsic_idx = np.arange(n_total)
else:
    hsic_idx = np.sort(rng.choice(n_total, size=HSIC_MAX_SAMPLES, replace=False))

X_hsic = X_numeric_full[hsic_idx]
x_is_acting_hsic = x_is_acting_full[hsic_idx]

print(f'Points disponibles : {n_total}')
print(f'Points utilisés pour HSIC : {len(hsic_idx)}')
pd.Series(hsic_idx, name='row_index').to_csv(OUTPUT_DIR / 'hsic_sample_indices.csv', index=False)


## 7. Random Forest supervisée et HSIC-ANOVA par sortie

In [ ]:
all_importance_tables = []
all_hsic_tables = []
summary_rows = []

for output in OUTPUT_COLS:
    print()
    print('=' * 90)
    print(f'Sortie MAELIA : {output}')
    print('=' * 90)

    y_full = df[output].to_numpy(dtype=float)
    y_hsic = y_full[hsic_idx]

    rf = RandomForestRegressor(
        n_estimators=RF_N_ESTIMATORS,
        max_depth=RF_MAX_DEPTH,
        min_samples_leaf=RF_MIN_SAMPLES_LEAF,
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )
    rf.fit(X_numeric_full, y_full)

    result = permutation_importance(
        rf,
        X_numeric_full,
        y_full,
        n_repeats=PERMUTATION_REPEATS,
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )

    theta = result.importances_mean * THETA_SCALE
    theta[result.importances_mean < THETA_IMPORTANCE_THRESHOLD] = 0.0

    importance_df = pd.DataFrame({
        'sortie': output,
        'variable': AGRI_FEATURES,
        'importance_moyenne': result.importances_mean,
        'importance_ecart_type': result.importances_std,
        'theta': theta,
        'conditionnelle': num_is_decreed,
        'categorielle': is_categorical,
        'frequence_active': x_is_acting_full.mean(axis=0),
    }).sort_values('importance_moyenne', ascending=False)
    all_importance_tables.append(importance_df)
    display(importance_df.head(12))

    filtered_results, global_hsic = hsic_anova_hierarchical(
        X=X_hsic,
        Y=y_hsic,
        x_is_acting=x_is_acting_hsic,
        num_is_decreed=num_is_decreed,
        is_categorical=is_categorical,
        theta_scales=theta,
        var_names=AGRI_FEATURES,
        max_order=HSIC_MAX_ORDER,
        use_smt_theta=True,
        use_kta=False,
    )

    hsic_df = pd.DataFrame(filtered_results)
    if not hsic_df.empty:
        hsic_df['sortie'] = output
        hsic_df['variables'] = hsic_df['combo'].apply(
            lambda combo: ' & '.join(AGRI_FEATURES[i] for i in combo)
        )
        hsic_df['global_var_pct'] = 100 * hsic_df['trace'] / global_hsic if global_hsic != 0 else np.nan
        adj_sum = hsic_df['adj_trace'].sum()
        hsic_df['intrinsic_var_pct'] = 100 * hsic_df['adj_trace'] / adj_sum if adj_sum != 0 else np.nan
        hsic_df = hsic_df[
            ['sortie', 'order', 'variables', 'global_var_pct', 'intrinsic_var_pct', 'p_A', 'trace', 'adj_trace']
        ]
    else:
        hsic_df = pd.DataFrame(columns=[
            'sortie', 'order', 'variables', 'global_var_pct', 'intrinsic_var_pct', 'p_A', 'trace', 'adj_trace'
        ])

    all_hsic_tables.append(hsic_df)
    display(hsic_df.head(20))

    summary_rows.append({
        'sortie': output,
        'n_total': n_total,
        'n_hsic': len(hsic_idx),
        'global_hsic': global_hsic,
        'n_terms_kept': len(hsic_df),
        'rf_importance_sum': float(np.nansum(result.importances_mean)),
        'theta_non_zero': int(np.sum(theta > 0)),
    })

importance_all = pd.concat(all_importance_tables, ignore_index=True)
hsic_all = pd.concat(all_hsic_tables, ignore_index=True)
hsic_summary = pd.DataFrame(summary_rows)

importance_all.to_csv(OUTPUT_DIR / 'rf_permutation_importance_real_maelia.csv', index=False)
hsic_all.to_csv(OUTPUT_DIR / 'hsic_anova_terms_real_maelia.csv', index=False)
hsic_summary.to_csv(OUTPUT_DIR / 'hsic_anova_summary_real_maelia.csv', index=False)

print()
print('Fichiers écrits :')
print(' -', OUTPUT_DIR / 'rf_permutation_importance_real_maelia.csv')
print(' -', OUTPUT_DIR / 'hsic_anova_terms_real_maelia.csv')
print(' -', OUTPUT_DIR / 'hsic_anova_summary_real_maelia.csv')


## 8. Résumé global

In [ ]:
display(hsic_summary)

top_terms = (
    hsic_all.sort_values(['sortie', 'intrinsic_var_pct'], ascending=[True, False])
    .groupby('sortie')
    .head(12)
)
display(top_terms)
